# 🏰 Deine Daten sind dein Burggraben

GPT-4o, Claude, Gemini — jeder kann diese Modelle nutzen. **Das Modell ist gemietet.** Was dich von der Konkurrenz unterscheidet, sind **deine Daten**.

In diesem Notebook:
1. Optimierung auf **verschiedenen Aufgaben** ausprobieren
2. Echte Ticket-Daten laden → generischer Prompt → mässiges Ergebnis
3. Mit deinen Daten tunen → deutlich besser
4. Verstehen: **Domain-Wissen = Wettbewerbsvorteil**


In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, pandas as pd
from dspy_tasks.config import configure_dspy
from dspy_tasks.tasks import get_task, list_tasks
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)


## 1. Welche Aufgaben gibt es?

Wir haben 20 verschiedene Aufgaben in 4 Schwierigkeitsstufen. Jede hat ein eigenes Dataset und eine Metrik. Hier eine Übersicht:


In [ ]:
tier_labels = {1: "Basics", 2: "Reasoning", 3: "Composition", 4: "Agentic"}

for tier in [1, 2, 3, 4]:
    tasks_in_tier = [t for t in list_tasks() if t.tier == tier]
    print(f"{'★' * tier}{'☆' * (4-tier)} Tier {tier} — {tier_labels[tier]}")
    for t in tasks_in_tier:
        print(f"  • {t.id:25s} {t.name}")
    print()


## 2. Optimierung auf einer beliebigen Aufgabe

Lass uns eine Aufgabe wählen und schauen, was der Optimizer daraus macht. Ändere den `TASK`-String und führe die Zelle erneut aus, um andere Aufgaben zu testen.


In [ ]:
# Ändere den Task hier (z.B. "sentiment", "math_word", "fact_verification")
TASK = "sentiment"

task = get_task(TASK)
print(f"📋 {task.name}")
print(f"   {task.description}")
print(f"   Schwierigkeit: {task.difficulty}")
print()

print(f"⏳ Optimiere {task.name}...\n")
result = run_optimization(TASK, "BootstrapFewShot", max_eval=8)

display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)


## 3. Jetzt mit deinen echten Daten

Generische Aufgaben wie Sentiment oder Mathe sind nett — aber der **echte Wert** zeigt sich bei **deinen eigenen Daten**.

Wir laden echte Support-Tickets aus dem Projekt. Jedes Ticket hat:
- **Zusammenfassung** — was ist das Problem?
- **Kategorie** — Network, Software, Hardware, ...
- **Priorität** — Critical, High, Medium, Low
- **Team** — welches Team kümmert sich?

Das Modell soll neue Tickets automatisch klassifizieren.


In [ ]:
task = get_task("ticket_routing")
examples = task.load_examples()

print(f"📊 {len(examples)} echte Tickets geladen\n")
for ex in examples[:5]:
    print(f"  📋 {str(ex.summary)[:80]}...")
    print(f"     → Kategorie: {ex.category} | Priorität: {ex.priority} | Team: {ex.assigned_group}")
    print()


## 4. Generischer Prompt → wie gut ist er?

Erst testen wir ohne Tuning — nur ein generischer Prompt: "Klassifiziere dieses Ticket."

Die Metrik bewertet drei Felder gleichzeitig:
- Priorität korrekt? → **40%** Gewicht (am wichtigsten!)
- Kategorie korrekt? → **35%** Gewicht
- Richtiges Team? → **25%** Gewicht


In [ ]:
print(f"⏳ Teste generischen Prompt auf {MODEL}...\n")

baseline = run_baseline("ticket_routing", max_eval=10)

display_score("Generischer Prompt (Baseline)", baseline.score)
display_results_table(baseline.individual_scores[:5])

print(f"\n👆 {baseline.score:.0%} — das Modell kennt eure Teams und Kategorien nicht!")


## 5. Domain-Tuning → der Unterschied

Jetzt lassen wir den Optimizer mit **deinen echten Ticket-Beispielen** trainieren. Gleiches Modell, gleiche Aufgabe — aber mit Domain-Wissen.


In [ ]:
print(f"⏳ Optimiere mit echten Ticket-Daten... (10-30 Sekunden)\n")

result = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=10)

display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)


## 6. Was du gerade gesehen hast

| | Generisch | Domain-getuned |
|---|---|---|
| Prompt | "Klassifiziere dieses Ticket" | Optimierter Prompt + echte Beispiele |
| Wissen | Keines über eure Teams/Kategorien | Lernt aus euren echten Tickets |
| Aufwand | 0 Sekunden | 10-30 Sekunden (einmalig) |

### 💡 Die Lektion

- **Das Modell ist gemietet** — GPT-4o kann jeder nutzen
- **Deine Daten gehören dir** — eure Tickets, eure Kategorien, eure Teams
- **Tuning macht den Unterschied** — messbar, reproduzierbar, einmalige Kosten
- **Das ist dein Burggraben** — kein Konkurrent kann eure Trainingsdaten kopieren


In [ ]:
display_insight("Dein Burggraben",
    f"Generischer Prompt: {result.baseline_score:.0%}. "
    f"Getuned mit DEINEN Daten: {result.optimized_score:.0%}. "
    "Das Modell ist gemietet — deine Daten sind es nicht.")


## ⏭️ Weiter geht's!

Deine Daten + Tuning = Burggraben. Aber können auch **Agenten** optimiert werden? Agenten die selbst entscheiden, welche Tools sie nutzen?

👉 **[Weiter zu Notebook: Agenten →](04_agents.ipynb)**
